# Rao (2021) — Novel Local Radiomic Bayesian Classifiers

**Paper:** *Novel Local Radiomic Bayesian Classifiers for Non-Invasive Prediction of MGMT Methylation Status in Glioblastoma*  
**Author:** Mihir Rao  
**arXiv:** 2112.03259

This notebook reproduces the **published workflow as closely as the paper allows**:

- BraTS 2021
- 577 patients with both segmentation and MGMT status
- 301 MGMT methylated / 276 unmethylated
- FLAIR only
- whole-tumor segmentation
- 93 PyRadiomics features
- Global radiomics
- Local radiomics in **3×3×3** neighborhoods
- tumor-containing crop resized to **33×33×33**
- local activation maps
- `%A = A / (A + N)`
- feature-wise Bayesian classifiers
- Gaussian Kernel Density Estimation (KDE)
- **10-fold stratified cross-validation**

## Important reproducibility note

The paper does **not** fully specify:
1. the exact numerical bandwidth grid/domain used in `GridSearchCV`;
2. every low-level PyRadiomics discretization/configuration value;
3. the exact interpolation settings used when resizing the crop to 33×33×33.

Those are therefore exposed as explicit parameters in this notebook and are labelled as implementation assumptions.

The notebook does **not** silently replace the paper's workflow with the thesis workflow.


## 1. Install dependencies


In [ ]:
# Run only if needed
# !pip install numpy pandas scipy SimpleITK pyradiomics scikit-learn matplotlib statsmodels


## 2. Imports

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt

from scipy import ndimage as ndi
from scipy.stats import ttest_ind

from radiomics import featureextractor

from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV,
    LeaveOneOut,
)
from sklearn.neighbors import KernelDensity
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)


## 3. Published experiment constants

The 93 features correspond exactly to the non-shape PyRadiomics groups:

- First-order: 18
- GLCM: 24
- GLRLM: 16
- GLSZM: 16
- GLDM: 14
- NGTDM: 5

Total:

\[
18+24+16+16+14+5 = 93
\]

The paper uses whole tumor = all provided tumor regions combined.


In [ ]:
RANDOM_STATE = 42

N_PATIENTS_EXPECTED = 577
EXPECTED_MGMT_1 = 301
EXPECTED_MGMT_0 = 276

LOCAL_CROP_SIZE = (33, 33, 33)  # x,y,z conceptually; handled as image size below
LOCAL_KERNEL_RADIUS = 1         # 3x3x3 neighborhood
LOCAL_MIN_TUMOR_VOXELS = 1     # at least 1 tumor voxel in the 27-voxel cube
N_SPLITS = 10
P_VALUE_THRESHOLD = 0.05

FEATURE_CLASSES = [
    "firstorder",
    "glcm",
    "glrlm",
    "glszm",
    "gldm",
    "ngtdm",
]

# Paper says KDE bandwidth is optimized with GridSearchCV over a manually-set domain,
# but the exact domain is not published.
# This grid is therefore an explicit implementation assumption.
KDE_BANDWIDTH_GRID = np.logspace(-2, 1, 40)

print("Expected PyRadiomics feature count: 93")


## 4. Input manifest

Prepare one CSV with:

```text
patient_id,flair_path,seg_path,mgmt
BraTS2021_00000,/path/..._flair.nii.gz,/path/..._seg.nii.gz,1
...
```

Where:
- `mgmt = 1` → methylated
- `mgmt = 0` → unmethylated
- `seg != 0` → whole tumor

This matches the paper's merging of Task 1 segmentation data and Task 2 MGMT labels by unique patient ID.


In [ ]:
MANIFEST_PATH = Path("/path/to/brats2021_577_manifest.csv")  # <-- change
OUTDIR = Path("outputs_rao2021")
OUTDIR.mkdir(parents=True, exist_ok=True)

# Reuse intermediate files when available.
REUSE_GLOBAL_FEATURES = True
REUSE_LOCAL_PERCENT_ACTIVATION = True

# Paper does not publish exact resize interpolation.
# Linear interpolation for FLAIR and nearest-neighbor for segmentation are explicit assumptions here.
FLAIR_INTERPOLATOR = sitk.sitkLinear
MASK_INTERPOLATOR = sitk.sitkNearestNeighbor

# PyRadiomics settings not explicitly locked by the paper.
# Keep defaults unless you have source code proving other values.
PYRAD_SETTINGS = {
    "normalize": False,
    "resampledPixelSpacing": None,
    "force2D": False,
}


## 5. Load and verify the 577-patient cohort

In [ ]:
def load_manifest(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)

    required = {"patient_id", "flair_path", "seg_path", "mgmt"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing manifest columns: {sorted(missing)}")

    df["patient_id"] = df["patient_id"].astype(str)
    df["mgmt"] = df["mgmt"].astype(int)

    if not set(df["mgmt"].unique()).issubset({0, 1}):
        raise ValueError("mgmt must contain only 0/1.")

    for c in ["flair_path", "seg_path"]:
        df[c] = df[c].map(lambda x: str(Path(x).expanduser().resolve()))
        bad = [p for p in df[c] if not Path(p).exists()]
        if bad:
            raise FileNotFoundError(f"{c}: missing file, example: {bad[0]}")

    if df["patient_id"].duplicated().any():
        raise ValueError("Duplicate patient_id values found.")

    return df


manifest = load_manifest(MANIFEST_PATH)

print("Patients:", len(manifest))
print(manifest["mgmt"].value_counts().sort_index())

if len(manifest) != N_PATIENTS_EXPECTED:
    warnings.warn(
        f"Paper cohort contains 577 patients; current manifest has {len(manifest)}."
    )

display(manifest.head())


## 6. Build the 93-feature PyRadiomics extractor

Shape features are intentionally excluded because the paper describes 93 first/second-order textural features.


In [ ]:
def build_93_extractor(voxel_based=False):
    extractor = featureextractor.RadiomicsFeatureExtractor(**PYRAD_SETTINGS)

    extractor.disableAllFeatures()
    for cls in FEATURE_CLASSES:
        extractor.enableFeatureClassByName(cls)

    extractor.disableAllImageTypes()
    extractor.enableImageTypeByName("Original")

    # For local maps PyRadiomics needs voxel-specific settings.
    if voxel_based:
        extractor.settings["kernelRadius"] = LOCAL_KERNEL_RADIUS
        extractor.settings["maskedKernel"] = False

    return extractor


def radiomics_feature_names_from_result(result):
    names = []
    for k in result:
        if k.startswith("original_"):
            names.append(k.replace("original_", ""))
    return names


## 7. Global radiomics — whole tumor

For each patient:
1. use FLAIR;
2. use whole-tumor mask (`seg != 0`);
3. extract 93 features;
4. obtain one 93-length vector.

The paper then Z-normalizes each feature using the cohort mean and standard deviation.


In [ ]:
def make_whole_tumor_mask(seg_img: sitk.Image) -> sitk.Image:
    arr = sitk.GetArrayFromImage(seg_img)
    mask_arr = (arr != 0).astype(np.uint8)

    mask = sitk.GetImageFromArray(mask_arr)
    mask.CopyInformation(seg_img)
    return mask


def extract_global_features(manifest_df: pd.DataFrame) -> pd.DataFrame:
    out_path = OUTDIR / "global_93_raw.csv"

    if REUSE_GLOBAL_FEATURES and out_path.exists():
        print("Reusing:", out_path)
        return pd.read_csv(out_path)

    extractor = build_93_extractor(voxel_based=False)
    rows = []

    for i, row in manifest_df.iterrows():
        pid = row["patient_id"]
        print(f"[Global {i+1}/{len(manifest_df)}] {pid}")

        flair = sitk.ReadImage(row["flair_path"])
        seg = sitk.ReadImage(row["seg_path"])
        mask = make_whole_tumor_mask(seg)

        result = extractor.execute(flair, mask)

        record = {
            "patient_id": pid,
            "mgmt": int(row["mgmt"]),
        }

        for k, v in result.items():
            if k.startswith("original_"):
                name = k.replace("original_", "")
                try:
                    record[name] = float(v)
                except Exception:
                    record[name] = np.nan

        rows.append(record)

    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False)

    feature_cols = [c for c in df.columns if c not in {"patient_id", "mgmt"}]
    print("Extracted global features:", len(feature_cols))

    return df


global_raw = extract_global_features(manifest)
display(global_raw.head())


## 8. Global Z-normalization and t-test feature selection

The paper describes:
- Z-normalizing a patient's raw feature value using the mean and SD of that feature across all patients;
- performing a t-test between MGMT 1 and MGMT 0;
- selecting `p < 0.05`;
- obtaining **46/93 significant global features**.

### Methodological note

This is reproduced as described. Under stricter modern ML practice, normalization and feature selection would normally be fit within each training fold to avoid leakage. This notebook preserves the paper workflow for comparison.


In [ ]:
def z_normalize_cohort(df: pd.DataFrame):
    meta = {"patient_id", "mgmt"}
    feature_cols = [c for c in df.columns if c not in meta]

    z = df.copy()
    stats = {}

    for c in feature_cols:
        mu = df[c].mean()
        sd = df[c].std(ddof=0)

        stats[c] = {"mean": float(mu), "sd": float(sd)}

        if sd > 0 and np.isfinite(sd):
            z[c] = (df[c] - mu) / sd
        else:
            z[c] = 0.0

    return z, stats


def ttest_select(df: pd.DataFrame, p_threshold=0.05):
    feature_cols = [c for c in df.columns if c not in {"patient_id", "mgmt"}]

    rows = []

    for c in feature_cols:
        x1 = df.loc[df["mgmt"] == 1, c].dropna().to_numpy()
        x0 = df.loc[df["mgmt"] == 0, c].dropna().to_numpy()

        stat, p = ttest_ind(
            x1,
            x0,
            equal_var=False,
            nan_policy="omit",
        )

        rows.append({
            "feature": c,
            "t_stat": stat,
            "p_value": p,
            "significant": bool(np.isfinite(p) and p < p_threshold),
        })

    result = pd.DataFrame(rows).sort_values("p_value")
    return result


global_z, global_z_stats = z_normalize_cohort(global_raw)
global_ttest = ttest_select(global_z, P_VALUE_THRESHOLD)

global_sig = global_ttest[global_ttest["significant"]]["feature"].tolist()

print("Global significant features:", len(global_sig))
display(global_ttest.head(20))

global_z.to_csv(OUTDIR / "global_93_z_normalized.csv", index=False)
global_ttest.to_csv(OUTDIR / "global_ttest_results.csv", index=False)


## 9. Crop whole-tumor region and resize to 33×33×33

The paper forms a minimum-size cube around the general tumor region and resizes it to a uniform **33×33×33** voxels.

Implementation here:
- determine the tight whole-tumor bounding box;
- expand shorter dimensions symmetrically to create a cube;
- crop FLAIR and segmentation;
- resample both to 33×33×33;
- FLAIR: linear interpolation;
- segmentation: nearest-neighbor interpolation.

The paper does not publish the exact interpolation choices, so these are explicit assumptions.


In [ ]:
def bbox_from_mask(mask_zyx):
    coords = np.argwhere(mask_zyx)
    if coords.size == 0:
        raise ValueError("Empty tumor mask.")

    z0, y0, x0 = coords.min(axis=0)
    z1, y1, x1 = coords.max(axis=0) + 1

    return (x0, y0, z0), (x1, y1, z1)


def cube_bounds(mask_zyx):
    (x0, y0, z0), (x1, y1, z1) = bbox_from_mask(mask_zyx)

    sx, sy, sz = x1 - x0, y1 - y0, z1 - z0
    side = max(sx, sy, sz)

    cx = (x0 + x1 - 1) / 2
    cy = (y0 + y1 - 1) / 2
    cz = (z0 + z1 - 1) / 2

    starts = [
        int(round(cx - (side - 1) / 2)),
        int(round(cy - (side - 1) / 2)),
        int(round(cz - (side - 1) / 2)),
    ]

    dims_xyz = [mask_zyx.shape[2], mask_zyx.shape[1], mask_zyx.shape[0]]

    for d in range(3):
        starts[d] = max(0, starts[d])
        if starts[d] + side > dims_xyz[d]:
            starts[d] = max(0, dims_xyz[d] - side)

    size = [
        min(side, dims_xyz[0] - starts[0]),
        min(side, dims_xyz[1] - starts[1]),
        min(side, dims_xyz[2] - starts[2]),
    ]

    return tuple(starts), tuple(size)


def crop_sitk(image, start_xyz, size_xyz):
    return sitk.RegionOfInterest(
        image,
        size=[int(v) for v in size_xyz],
        index=[int(v) for v in start_xyz],
    )


def resample_to_33(image, is_mask=False):
    old_size = np.array(image.GetSize(), dtype=float)
    old_spacing = np.array(image.GetSpacing(), dtype=float)

    new_size = np.array([33, 33, 33], dtype=int)
    new_spacing = old_spacing * old_size / new_size

    resampler = sitk.ResampleImageFilter()
    resampler.SetSize([int(v) for v in new_size])
    resampler.SetOutputSpacing([float(v) for v in new_spacing])
    resampler.SetOutputOrigin(image.GetOrigin())
    resampler.SetOutputDirection(image.GetDirection())
    resampler.SetTransform(sitk.Transform())
    resampler.SetDefaultPixelValue(0)

    if is_mask:
        resampler.SetInterpolator(MASK_INTERPOLATOR)
        out = resampler.Execute(sitk.Cast(image, sitk.sitkUInt8))
    else:
        resampler.SetInterpolator(FLAIR_INTERPOLATOR)
        out = resampler.Execute(sitk.Cast(image, sitk.sitkFloat32))

    return out


def prepare_local_33(flair_path, seg_path):
    flair = sitk.ReadImage(str(flair_path))
    seg = sitk.ReadImage(str(seg_path))

    seg_arr = sitk.GetArrayFromImage(seg)
    tumor = seg_arr != 0

    start, size = cube_bounds(tumor)

    flair_crop = crop_sitk(flair, start, size)
    seg_crop = crop_sitk(seg, start, size)

    flair_33 = resample_to_33(flair_crop, is_mask=False)

    tumor_crop_arr = (sitk.GetArrayFromImage(seg_crop) != 0).astype(np.uint8)
    tumor_crop = sitk.GetImageFromArray(tumor_crop_arr)
    tumor_crop.CopyInformation(seg_crop)

    tumor_33 = resample_to_33(tumor_crop, is_mask=True)

    return flair_33, tumor_33


## 10. Centers eligible for local 3×3×3 radiomics

A local 3×3×3 cube is processed when it contains at least **1 tumor voxel out of 27**.

We identify all eligible center voxels using a 3×3×3 convolution over the resized tumor mask.


In [ ]:
def eligible_center_mask(tumor_33: sitk.Image) -> sitk.Image:
    arr = sitk.GetArrayFromImage(tumor_33).astype(np.uint8)

    local_count = ndi.convolve(
        arr,
        np.ones((3, 3, 3), dtype=np.uint8),
        mode="constant",
        cval=0,
    )

    eligible = (local_count >= LOCAL_MIN_TUMOR_VOXELS).astype(np.uint8)

    # Exclude border centers because a full 3x3x3 cube does not exist there.
    eligible[[0, -1], :, :] = 0
    eligible[:, [0, -1], :] = 0
    eligible[:, :, [0, -1]] = 0

    mask = sitk.GetImageFromArray(eligible)
    mask.CopyInformation(tumor_33)
    return mask


## 11. Local 93-feature value maps

Instead of calling PyRadiomics separately tens of thousands of times per patient, this notebook uses PyRadiomics' `voxelBased=True` mode with `kernelRadius=1`.

This computes the same *type* of local 3×3×3 radiomic neighborhood much more efficiently.

### Reproducibility qualification

The paper describes a literal sliding 3×3×3 cube implementation but does not publish source code. PyRadiomics voxel-based extraction is therefore a practical implementation approximation, while preserving:
- the same PyRadiomics feature definitions;
- local 3×3×3 neighborhoods;
- the same tumor-containing-cube eligibility rule.


In [ ]:
def extract_local_feature_maps(flair_33, tumor_33):
    center_mask = eligible_center_mask(tumor_33)

    extractor = build_93_extractor(voxel_based=True)

    result = extractor.execute(
        flair_33,
        center_mask,
        label=1,
        voxelBased=True,
    )

    maps = {}
    for k, v in result.items():
        if k.startswith("original_") and isinstance(v, sitk.Image):
            maps[k.replace("original_", "")] = v

    return maps, center_mask


## 12. Activation maps and `%A`

For every feature, Rao defines a local cube as:
- **activated** if local feature value ≥ global median value for that feature across patients;
- **non-activated** otherwise.

For each patient:

\[
\%A = \frac{A}{A+N}
\]

where `A` is the number of activated local positions and `N` is the number of non-activated local positions.

The global medians are calculated from the global 93-feature cohort table, as described in the paper.


In [ ]:
def feature_global_medians(global_df):
    medians = {}
    for c in global_df.columns:
        if c not in {"patient_id", "mgmt"}:
            medians[c] = float(global_df[c].median())
    return medians


GLOBAL_MEDIANS = feature_global_medians(global_raw)


def get_map_values_at_mask(feature_img, mask_img):
    mask_arr = sitk.GetArrayFromImage(mask_img).astype(bool)

    # Feature-map geometry can be cropped by PyRadiomics.
    # Map every active center via physical coordinates.
    center_indices_zyx = np.argwhere(mask_arr)

    vals = []

    for z, y, x in center_indices_zyx:
        point = mask_img.TransformIndexToPhysicalPoint((int(x), int(y), int(z)))

        try:
            idx = feature_img.TransformPhysicalPointToIndex(point)
        except RuntimeError:
            continue

        size = feature_img.GetSize()
        if any(idx[d] < 0 or idx[d] >= size[d] for d in range(3)):
            continue

        vals.append(float(feature_img.GetPixel(idx)))

    return np.asarray(vals, dtype=float)


def extract_local_percent_activation(manifest_df, global_medians):
    out_path = OUTDIR / "local_93_percent_activation.csv"

    if REUSE_LOCAL_PERCENT_ACTIVATION and out_path.exists():
        print("Reusing:", out_path)
        return pd.read_csv(out_path)

    rows = []

    for i, row in manifest_df.iterrows():
        pid = row["patient_id"]
        print(f"[Local {i+1}/{len(manifest_df)}] {pid}")

        flair_33, tumor_33 = prepare_local_33(
            row["flair_path"],
            row["seg_path"],
        )

        fmap_dict, center_mask = extract_local_feature_maps(
            flair_33,
            tumor_33,
        )

        record = {
            "patient_id": pid,
            "mgmt": int(row["mgmt"]),
        }

        for feat, fmap in fmap_dict.items():
            values = get_map_values_at_mask(fmap, center_mask)
            values = values[np.isfinite(values)]

            if values.size == 0 or feat not in global_medians:
                record[feat] = np.nan
                continue

            threshold = global_medians[feat]
            activated = np.count_nonzero(values >= threshold)

            record[feat] = activated / values.size

        rows.append(record)

    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False)

    feature_cols = [c for c in df.columns if c not in {"patient_id", "mgmt"}]
    print("Local activation features:", len(feature_cols))

    return df


local_activation = extract_local_percent_activation(
    manifest,
    GLOBAL_MEDIANS,
)

display(local_activation.head())


## 13. Local t-test feature selection

The paper reports:
- **62/93** local `%A` features significant at `p < 0.05`.

This cell reproduces that test.


In [ ]:
local_ttest = ttest_select(
    local_activation,
    P_VALUE_THRESHOLD,
)

local_sig = local_ttest[
    local_ttest["significant"]
]["feature"].tolist()

print("Local significant features:", len(local_sig))
display(local_ttest.head(20))

local_ttest.to_csv(
    OUTDIR / "local_ttest_results.csv",
    index=False,
)


## 14. KDE Bayesian classifier

For a single feature `x`, the classifier compares:

\[
p(x|MGMT=1)P(MGMT=1)
\]

with

\[
p(x|MGMT=0)P(MGMT=0)
\]

Two separate Gaussian KDE models are fit:
- one for MGMT 1;
- one for MGMT 0.

The KDE bandwidth is optimized inside each training fold using `GridSearchCV`, matching the paper's stated method.

### Unspecified paper detail

The exact manually-set bandwidth search domain is not reported. This notebook uses `KDE_BANDWIDTH_GRID` defined above.


In [ ]:
def fit_best_kde_1d(values):
    values = np.asarray(values, dtype=float).reshape(-1, 1)

    # With small 1-D training subsets, leave-one-out gives stable bandwidth selection
    # without introducing a fixed arbitrary number of inner folds.
    search = GridSearchCV(
        KernelDensity(kernel="gaussian"),
        {"bandwidth": KDE_BANDWIDTH_GRID},
        cv=LeaveOneOut(),
        n_jobs=-1,
    )

    search.fit(values)
    return search.best_estimator_, float(search.best_params_["bandwidth"])


def bayes_kde_predict(train_x, train_y, test_x):
    train_x = np.asarray(train_x, dtype=float)
    train_y = np.asarray(train_y, dtype=int)
    test_x = np.asarray(test_x, dtype=float)

    x1 = train_x[train_y == 1]
    x0 = train_x[train_y == 0]

    kde1, bw1 = fit_best_kde_1d(x1)
    kde0, bw0 = fit_best_kde_1d(x0)

    p1 = len(x1) / len(train_y)
    p0 = len(x0) / len(train_y)

    log1 = kde1.score_samples(test_x.reshape(-1, 1)) + np.log(p1)
    log0 = kde0.score_samples(test_x.reshape(-1, 1)) + np.log(p0)

    pred = (log1 >= log0).astype(int)

    # Stable posterior-like score for optional ROC-AUC.
    delta = np.clip(log1 - log0, -50, 50)
    prob1 = 1.0 / (1.0 + np.exp(-delta))

    return pred, prob1, bw1, bw0


## 15. 10-fold stratified CV for each feature

Rao builds **one separate Bayesian classifier per significant feature**.

Thus:
- Global → one classifier for each of the 46 significant global features;
- Local → one classifier for each of the 62 significant local activation features.

The reported comparison is the mean CV accuracy per feature.


In [ ]:
def evaluate_featurewise_bayes(
    df,
    features,
    feature_source,
):
    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    y = df["mgmt"].to_numpy(dtype=int)
    all_rows = []

    for feature in features:
        x = df[feature].to_numpy(dtype=float)

        # Keep patients with finite value for this feature.
        valid = np.isfinite(x)
        xv = x[valid]
        yv = y[valid]

        fold_acc = []

        for fold, (tr, va) in enumerate(
            skf.split(xv.reshape(-1, 1), yv),
            start=1,
        ):
            pred, prob, bw1, bw0 = bayes_kde_predict(
                xv[tr],
                yv[tr],
                xv[va],
            )

            acc = accuracy_score(yv[va], pred)
            fold_acc.append(acc)

            row = {
                "source": feature_source,
                "feature": feature,
                "fold": fold,
                "accuracy": acc,
                "precision": precision_score(
                    yv[va], pred, zero_division=0
                ),
                "recall": recall_score(
                    yv[va], pred, zero_division=0
                ),
                "f1": f1_score(
                    yv[va], pred, zero_division=0
                ),
                "roc_auc": (
                    roc_auc_score(yv[va], prob)
                    if len(np.unique(yv[va])) == 2
                    else np.nan
                ),
                "bandwidth_mgmt1": bw1,
                "bandwidth_mgmt0": bw0,
            }

            all_rows.append(row)

    fold_df = pd.DataFrame(all_rows)

    summary = (
        fold_df
        .groupby(["source", "feature"])
        .agg(
            mean_accuracy=("accuracy", "mean"),
            sd_accuracy=("accuracy", "std"),
            mean_precision=("precision", "mean"),
            mean_recall=("recall", "mean"),
            mean_f1=("f1", "mean"),
            mean_roc_auc=("roc_auc", "mean"),
        )
        .reset_index()
        .sort_values(
            "mean_accuracy",
            ascending=False,
        )
    )

    return fold_df, summary


## 16. Run Global Bayesian classifiers

In [ ]:
global_fold_results, global_cv_summary = evaluate_featurewise_bayes(
    global_z,
    global_sig,
    "global",
)

global_fold_results.to_csv(
    OUTDIR / "global_bayes_10fold_per_fold.csv",
    index=False,
)

global_cv_summary.to_csv(
    OUTDIR / "global_bayes_10fold_summary.csv",
    index=False,
)

display(global_cv_summary.head(20))


## 17. Run Local Bayesian classifiers

In [ ]:
local_fold_results, local_cv_summary = evaluate_featurewise_bayes(
    local_activation,
    local_sig,
    "local",
)

local_fold_results.to_csv(
    OUTDIR / "local_bayes_10fold_per_fold.csv",
    index=False,
)

local_cv_summary.to_csv(
    OUTDIR / "local_bayes_10fold_summary.csv",
    index=False,
)

display(local_cv_summary.head(20))


## 18. Compare features significant in both Global and Local approaches

The paper's Figure 11 compares common significant features and reports the percentage change in mean accuracy from Global to Local:

\[
\%\Delta Accuracy =
\frac{Acc_{local}-Acc_{global}}{Acc_{global}}\times100
\]


In [ ]:
g = global_cv_summary[
    ["feature", "mean_accuracy"]
].rename(
    columns={"mean_accuracy": "global_mean_accuracy"}
)

l = local_cv_summary[
    ["feature", "mean_accuracy"]
].rename(
    columns={"mean_accuracy": "local_mean_accuracy"}
)

comparison = g.merge(
    l,
    on="feature",
    how="inner",
)

comparison["percent_change_global_to_local"] = (
    (
        comparison["local_mean_accuracy"]
        - comparison["global_mean_accuracy"]
    )
    / comparison["global_mean_accuracy"]
    * 100.0
)

comparison = comparison.sort_values(
    "percent_change_global_to_local",
    ascending=False,
)

comparison.to_csv(
    OUTDIR / "global_vs_local_common_features.csv",
    index=False,
)

display(comparison)


## 19. Reproduce the paper-style result counts

In [ ]:
paper_check = pd.DataFrame({
    "quantity": [
        "patients",
        "MGMT methylated",
        "MGMT unmethylated",
        "global features extracted",
        "global significant p<0.05",
        "local features extracted",
        "local significant p<0.05",
        "common significant features",
    ],
    "observed": [
        len(manifest),
        int((manifest["mgmt"] == 1).sum()),
        int((manifest["mgmt"] == 0).sum()),
        len([c for c in global_raw.columns if c not in {"patient_id", "mgmt"}]),
        len(global_sig),
        len([c for c in local_activation.columns if c not in {"patient_id", "mgmt"}]),
        len(local_sig),
        len(comparison),
    ],
    "paper_reference": [
        577,
        301,
        276,
        93,
        46,
        93,
        62,
        "see paper Figure/Table",
    ],
})

display(paper_check)

paper_check.to_csv(
    OUTDIR / "paper_reproduction_check.csv",
    index=False,
)


## 20. Optional plots similar to Figures 9–11


In [ ]:
plt.figure(figsize=(12, 5))
tmp = global_cv_summary.sort_values("mean_accuracy", ascending=False)
plt.bar(range(len(tmp)), tmp["mean_accuracy"])
plt.xlabel("Significant global radiomic feature")
plt.ylabel("Mean 10-fold CV accuracy")
plt.title("Global Radiomic Bayesian Classifiers")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
tmp = local_cv_summary.sort_values("mean_accuracy", ascending=False)
plt.bar(range(len(tmp)), tmp["mean_accuracy"])
plt.xlabel("Significant local radiomic feature")
plt.ylabel("Mean 10-fold CV accuracy")
plt.title("Local Radiomic Bayesian Classifiers")
plt.tight_layout()
plt.show()

if len(comparison):
    plt.figure(figsize=(12, 5))
    tmp = comparison.sort_values(
        "percent_change_global_to_local",
        ascending=False,
    )
    plt.bar(
        range(len(tmp)),
        tmp["percent_change_global_to_local"],
    )
    plt.xlabel("Feature significant in both approaches")
    plt.ylabel("% change in mean accuracy")
    plt.title("Global → Local Bayesian Accuracy Change")
    plt.tight_layout()
    plt.show()


## 21. Outputs

The notebook creates:

- `global_93_raw.csv`
- `global_93_z_normalized.csv`
- `global_ttest_results.csv`
- `local_93_percent_activation.csv`
- `local_ttest_results.csv`
- `global_bayes_10fold_per_fold.csv`
- `global_bayes_10fold_summary.csv`
- `local_bayes_10fold_per_fold.csv`
- `local_bayes_10fold_summary.csv`
- `global_vs_local_common_features.csv`
- `paper_reproduction_check.csv`

## Expected checkpoints from Rao (2021)

A reproduction close to the paper should first verify:

- 577 patients
- 301 MGMT 1
- 276 MGMT 0
- 93 global features
- **46 global features with p < 0.05**
- 93 local activation features
- **62 local features with p < 0.05**

If the 46/62 counts differ materially, do **not** immediately tune the model to force agreement. First inspect:
- PyRadiomics version/settings;
- local crop/resizing;
- image discretization defaults;
- interpolation;
- exact Task 1↔Task 2 patient matching.

## Comparison with the thesis

This Rao notebook predicts **MGMT at patient level**.  
The thesis predicts **Tumor/Non-tumor at voxel level**.

Even though both use FLAIR + local PyRadiomics and 577 patients, their Accuracy/AUC values must not be ranked as if they were the same task.
